# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [126]:
import os
import json
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
import gradio as gr
import sqlite3
import base64
from io import BytesIO
from PIL import Image

In [ ]:
#loading api key

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

OpenAI API Key exists and begins sk-proj-


In [128]:
system_message = """
You are a helpful assistant for an application called FoodAI. Your job is to give ideas for dinner based on the cuisine a user wants. Strictly, If the tool call does not contain the cuisine or a recipe for that cuisine, say I am not sure what you can make
Give short, courteous answers, no more than 1 sentence.
Always be accurate. 
"""

In [129]:
food = {
    "name": "get_food_ideas",
    "description": "Get the food item based on cuisine",
    "parameters": {
        "type": "object",
        "properties": {
            "cuisine": {
                "type": "string",
                "description": "The cuisine that the customer wants to try",
            },
        },
        "required": ["cuisine"],
        "additionalProperties": False
    }
}

update_food = {
    "name": "update_food_ideas",
    "description": "Add new cuisines with food ideas",
    "parameters": {
        "type": "object",
        "properties": {
            "cuisine": {
                "type": "string",
                "description": "The cuisine the user wants to have",
            },
            "food_item": {
                "type": "string",
                "description": "the food item for a specific cuisine",
            }
        },
        "required": ["cuisine", "food_item"],
        "additionalProperties": False
    }
}

add_food = {
    "name": "add_cuisine_food",
    "description": "Add new cuisines with food ideas",
    "parameters": {
        "type": "object",
        "properties": {
            "cuisine": {
                "type": "string",
                "description": "The cuisine the user wants to have",
            },
            "food_item": {
                "type": "string",
                "description": "the food item for a specific cuisine",
            }
        },
        "required": ["cuisine", "food_item"],
        "additionalProperties": False
    }
}

In [130]:
tools = [{"type": "function", "function": food}, {"type": "function", "function": update_food}, {"type": "function", "function": add_food}]

In [131]:
DB = "foods.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS food (cuisine TEXT PRIMARY KEY, food_item TEXT)')
    conn.commit()

In [132]:
def get_food_ideas(cuisine):
    print(f"DATABASE TOOL CALLED: Getting food for {cuisine}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT food_item FROM food WHERE cuisine = ?', (cuisine.lower(),))
        result = cursor.fetchone()
        return f"A good food item for {cuisine} is ${result[0]}" if result else "No price data available for this city"

In [133]:
def update_food_ideas(cuisine, food_item):
    print(f"DATABASE TOOL CALLED: Updating food item for {cuisine} to {food_item}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("UPDATE food SET food_item = ? WHERE cuisine = ?",(food_item, cuisine))
        result = cursor.fetchone()
        return f"Food item for {cuisine} updated to {food_item}"

In [134]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image

In [146]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cuisines = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_food_ideas":
            arguments = json.loads(tool_call.function.arguments)
            cuisine = arguments.get('cuisine')
            cuisines.append(cuisine)
            food_item = get_food_ideas(cuisine)
            responses.append({
                "role": "tool",
                "content": food_item,
                "tool_call_id": tool_call.id
            })
        if tool_call.function.name == "update_food":
            arguments = json.loads(tool_call.function.arguments)
            cuisine = arguments.get('cuisine')
            food = arguments.get('food')
            food_update = update_food_ideas(cuisine, food)
            responses.append({
                "role": "tool",
                "content": food_update,
                "tool_call_id": tool_call.id
            })
        if tool_call.function.name == "add_cuisine_food":
            arguments = json.loads(tool_call.function.arguments)
            cuisine = arguments.get('cuisine')
            food_item = arguments.get('food_item')
            add_cuisine_food(cuisine, food_item)
            responses.append({
                "role": "tool",
                "content": food_item,
                "tool_call_id": tool_call.id
            })
    return responses, cuisines


In [136]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

In [137]:
def artist(food_item):
    response = openai.images.generate(
        model="gpt-image-1",
        prompt=f"An image representing {food_item}, in a vibrant pop-art style",
        size="1024x1024",
    )

    image_bytes = base64.b64decode(response.data[0].b64_json)
    return Image.open(BytesIO(image_bytes))

In [144]:
def add_cuisine_food(cuisine, food):
    print(f"DATABASE TOOL CALLED: Adding food item for {cuisine} to {food}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO food (cuisine, food_item) VALUES (?, ?) ON CONFLICT(cuisine) DO UPDATE SET food_item = ?', (cuisine.lower(), food.lower(), food.lower()))
        conn.commit()

In [139]:
food_items = {"italian":"aglio e olio", "indian": "paneer bhurji", "japanese": "ramen", "chinese": "chowmein"}
for cuisine, food in food_items.items():
    add_cuisine_food(cuisine, food)

In [ ]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Adding food item for mexican to tacos
DATABASE TOOL CALLED: Getting food for mexican
